# Krishimitra: Paddy Disease Classifier (EfficientNetV2-S)

Trains a 10-class rice disease classifier on the **Paddy Doctor** dataset. It then calibrates the model, evaluates it once on a held-out test set, and exports it to **ONNX** for the `/health/diagnose` API.

**How to use**
1. Edit **only** the config cell below (VRAM table inside).
2. *Kernel → Restart & Run All*.
3. When it finishes, send back `models/disease_classifier_v1.zip` and this executed notebook.

Every step is explained in plain language in **`GUIDE.md`**. The section numbers below match the plan and the guide.

| § | Section | GUIDE.md |
|---|---|---|
| 0 | Config | — |
| 1 | Environment check | — |
| 2 | Data | §2–4 |
| 3 | Transforms | §5 |
| 4 | Model | §1 |
| 5 | Training | §6 |
| 6 | Calibration | §8 |
| 7 | Test evaluation (run once) | §7 |
| 8 | Error analysis | §7 |
| 9 | Export | §9 |

## 0. Config: the only cell you should edit

In [ ]:
import os
from pathlib import Path

# ---------------- paths (relative to the training/ folder) ----------------
DATA_DIR   = Path(os.environ.get("KRISHI_DATA_DIR", "data/paddy"))   # contains train.csv + train_images/
SPLITS_CSV = Path(os.environ.get("KRISHI_SPLITS", "splits.csv"))      # made by make_split.py - do NOT regenerate
OUT_DIR    = Path("outputs")                                          # checkpoints + plots (git-ignored)
EXPORT_DIR = Path("../models/disease_classifier_v1")                  # final deliverable

# ---------------- model + image size ----------------
MODEL_NAME = "tf_efficientnetv2_s.in21k_ft_in1k"
IMG_SIZE   = 384
BATCH      = 16      # images per forward pass (limited by VRAM)
ACCUM_STEPS = 2      # effective batch = BATCH * ACCUM_STEPS = 32
# VRAM guide (IMG_SIZE / BATCH), keep BATCH * ACCUM_STEPS ~ 32:
#    6 GB  -> 320 / 12  (ACCUM_STEPS=3)
#    8 GB  -> 384 / 16  (ACCUM_STEPS=2)   <- default
#   12 GB  -> 384 / 24  (ACCUM_STEPS=1)
#   16 GB+ -> 448 / 32  (ACCUM_STEPS=1)
# "CUDA out of memory" -> go one row up.

# ---------------- schedule ----------------
EPOCHS_HEAD   = 2       # Stage A: train only the new head
EPOCHS_FT     = 25      # Stage B: fine-tune everything (early stopping usually ends it sooner)
LR_STAGE_A    = 1e-3    # head LR in stage A
LR_HEAD       = 3e-4    # head LR in stage B
LR_BACKBONE   = 1e-4    # backbone LR in stage B
WEIGHT_DECAY  = 0.05
WARMUP_EPOCHS = 1
LABEL_SMOOTHING = 0.1
MIXUP_PROB    = 0.3     # fraction of batches that get Mixup/CutMix (0 = off)
PATIENCE      = 6       # early stopping on val macro-F1
NUM_WORKERS   = None    # None = automatic
SEED          = 42
RESUME        = True    # continue from outputs/checkpoints/last.pt if it exists

# ---------------- smoke test (CPU, ~5-10 min) ----------------
SMOKE_TEST = True      # True = tiny run on 200 images to prove every cell works
SMOKE_TEST = SMOKE_TEST or os.environ.get("KRISHI_SMOKE") == "1"

if SMOKE_TEST:
    MODEL_NAME, IMG_SIZE, BATCH, ACCUM_STEPS = "efficientnet_b0", 128, 8, 1
    EPOCHS_HEAD, EPOCHS_FT, WARMUP_EPOCHS = 1, 1, 0
    OUT_DIR    = Path("outputs/smoke")
    EXPORT_DIR = Path("outputs/smoke/disease_classifier_v1")   # never overwrite the real model
    print("*** SMOKE TEST MODE: results are meaningless, this only checks the pipeline ***")

CKPT_DIR = OUT_DIR / "checkpoints"
for d in (OUT_DIR, CKPT_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Environment check

In [ ]:
import json, math, platform, shutil, sys, time
import numpy as np, pandas as pd, torch, timm, torchvision
import matplotlib.pyplot as plt
import krishi_data as kd

kd.seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__} | timm {timm.__version__}")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  |  VRAM: {props.total_memory / 1024**3:.1f} GB")
    torch.backends.cudnn.benchmark = True        # faster convolutions for fixed image size
else:
    msg = "NO CUDA GPU FOUND - training will run on the CPU."
    if not SMOKE_TEST:
        msg += ("\n    A full run on CPU would take DAYS. Install the CUDA build of PyTorch"
                "\n    (see RUN_ON_GPU.md), or set SMOKE_TEST = True.")
    print("!" * 70 + "\n!!  " + msg + "\n" + "!" * 70)

## 2. Data
Downloads Paddy Doctor if it's missing (this needs `kaggle.json`; see GUIDE §2). It then reads the **fixed** split from `splits.csv`, which was made once by `make_split.py`.

In [ ]:
if not (DATA_DIR / "train.csv").exists():
    print(f"{DATA_DIR / 'train.csv'} not found -> downloading with the Kaggle API (~1 GB)...")
    import zipfile
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
    except Exception as e:
        raise RuntimeError("Kaggle API not set up: put kaggle.json in ~/.kaggle/ and accept the "
                           "competition rules (GUIDE.md section 2).") from e
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    api.competition_download_files("paddy-disease-classification", path=str(DATA_DIR.parent), quiet=False)
    with zipfile.ZipFile(DATA_DIR.parent / "paddy-disease-classification.zip") as z:
        z.extractall(DATA_DIR)

splits = pd.read_csv(SPLITS_CSV)
missing = [p for p in splits["path"] if not (DATA_DIR / p).exists()]
assert not missing, f"{len(missing)} images in splits.csv are missing under {DATA_DIR}, e.g. {missing[:3]}"
assert set(splits["label"]) <= set(kd.CLASS_NAMES), "unknown label in splits.csv"

if SMOKE_TEST:   # ~200 images: 14 train / 3 val / 3 test per class
    per = {"train": 14, "val": 3, "test": 3}
    splits = pd.concat([splits[splits.split == s].sample(frac=1, random_state=SEED).groupby("label").head(k)
                        for s, k in per.items()], ignore_index=True)

counts = splits.pivot_table(index="label", columns="split", values="image_id", aggfunc="count", fill_value=0)
counts = counts.reindex(index=kd.CLASS_NAMES, columns=["train", "val", "test"], fill_value=0)
counts["total"] = counts.sum(1)
print(counts.to_string(), "\n\nTOTAL", counts.sum().to_dict())

counts[["train", "val", "test"]].plot.barh(stacked=True, figsize=(8, 4), title="Images per class and split")
plt.tight_layout(); plt.savefig(OUT_DIR / "class_counts.png", dpi=110); plt.show()

In [ ]:
# Variety vs disease: a strong link could let the model "cheat" by recognising the variety.
if "variety" in splits:
    print(pd.crosstab(splits["label"], splits["variety"]).reindex(kd.CLASS_NAMES).fillna(0).astype(int).to_string())

In [ ]:
# Sample grid: 4 random photos per class. Can YOU tell them apart? Any obvious mislabels?
from PIL import Image
n_per = 4
fig, axes = plt.subplots(len(kd.CLASS_NAMES), n_per, figsize=(n_per * 2.2, len(kd.CLASS_NAMES) * 2.2))
for r, cls in enumerate(kd.CLASS_NAMES):
    rows = splits[splits.label == cls]
    rows = rows.sample(min(n_per, len(rows)), random_state=SEED)
    for c in range(n_per):
        ax = axes[r, c]; ax.axis("off")
        if c < len(rows):
            ax.imshow(Image.open(DATA_DIR / rows.iloc[c]["path"]).convert("RGB"))
        if c == 0:
            ax.set_title(cls, fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(OUT_DIR / "sample_grid.png", dpi=90); plt.show()

## 3. Transforms
- **Train:** random crop, flips, rotation, colour jitter, blur and JPEG compression. These mimic real phone and WhatsApp photos.
- **Eval:** resize and centre crop **only**.

Normalisation uses the mean and std **this backbone was pretrained with**, which is not always ImageNet's.

In [ ]:
MEAN, STD = kd.model_norm_stats(MODEL_NAME)
print(f"normalisation for {MODEL_NAME}: mean={MEAN} std={STD}")

loaders = kd.make_loaders(splits, DATA_DIR, IMG_SIZE, BATCH, num_workers=NUM_WORKERS, mean=MEAN, std=STD)
print({k: f"{len(v.dataset)} images / {len(v)} batches" for k, v in loaders.items()})

def denorm(t):
    """Undo normalisation so a tensor can be displayed."""
    return (t * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0)

# Same photo, 8 random training augmentations (+ the eval version first)
row = splits[splits.split == "train"].iloc[0]
img = Image.open(DATA_DIR / row["path"]).convert("RGB")
t_train, t_eval = kd.build_transforms(IMG_SIZE, True, MEAN, STD), kd.build_transforms(IMG_SIZE, False, MEAN, STD)
fig, axes = plt.subplots(1, 9, figsize=(18, 2.4))
axes[0].imshow(denorm(t_eval(img))); axes[0].set_title("eval", fontsize=9)
for ax in axes[1:]:
    ax.imshow(denorm(t_train(img))); ax.set_title("train aug", fontsize=9)
for ax in axes: ax.axis("off")
plt.suptitle(row["label"]); plt.tight_layout(); plt.savefig(OUT_DIR / "augmentations.png", dpi=90); plt.show()

## 4. Model: an ImageNet-pretrained backbone with a new 10-class head

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(kd.CLASS_NAMES),
                          drop_rate=0.3, drop_path_rate=0.2).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.get_classifier().parameters())
print(f"{MODEL_NAME}: {n_total / 1e6:.1f} M parameters ({n_head:,} in the new head)")

## 5. Training
- **Stage A:** the backbone is frozen and only the head trains.
- **Stage B:** the whole network is fine-tuned with AdamW, warmup and a cosine schedule, label smoothing, light Mixup/CutMix, AMP and gradient accumulation.

After every epoch the notebook:
- measures **val macro-F1**,
- keeps `best.pt`,
- writes a resumable `last.pt`.

If the session dies, just **Run All again**. With `RESUME=True` it continues where it stopped.

In [ ]:
from dataclasses import asdict
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy

ckpt_last, ckpt_best = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt"
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
early = kd.EarlyStopping(patience=PATIENCE)
history = []
resume = None
if RESUME and ckpt_last.exists():
    resume = kd.load_checkpoint(ckpt_last, map_location=device, model=model)
    history, early = resume["history"], kd.EarlyStopping(**resume["early"])
    print(f"Resuming: stage {resume['stage']}, finished epoch {resume['epoch'] + 1}, best val macro-F1 {early.best:.4f}")
    if resume.get("done"):
        print("Training already finished - skipping to evaluation.")

def run_stage(stage, epochs, lr_head, lr_backbone):
    """Train one stage, with checkpointing / resume / early stopping."""
    global early
    if epochs <= 0:
        return
    if resume is not None and (resume.get("done") or (stage == "A" and resume["stage"] == "B")):
        return                                           # this stage already completed earlier
    kd.set_backbone_trainable(model, trainable=(stage == "B"))
    opt = kd.build_optimizer(model, lr_head, lr_backbone, WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(loaders["train"]) / ACCUM_STEPS)
    warmup = steps_per_epoch * WARMUP_EPOCHS if stage == "B" else 0
    sched = kd.cosine_with_warmup(opt, steps_per_epoch * epochs, warmup)
    if stage == "B" and MIXUP_PROB > 0:
        mixup = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=MIXUP_PROB,
                      label_smoothing=LABEL_SMOOTHING, num_classes=len(kd.CLASS_NAMES))
        criterion = SoftTargetCrossEntropy()
    else:
        mixup, criterion = None, torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    start = 0
    if resume is not None and resume["stage"] == stage:
        kd.load_checkpoint(ckpt_last, map_location=device, optimizer=opt, scheduler=sched, scaler=scaler)
        start = resume["epoch"] + 1
    elif stage == "B":
        early.bad_epochs = 0                              # patience counts only within stage B

    for epoch in range(start, epochs):
        t0 = time.time()
        tr = kd.train_one_epoch(model, loaders["train"], opt, sched, scaler, criterion, device,
                                accum_steps=ACCUM_STEPS, mixup_fn=mixup)
        val_logits, val_labels = kd.predict(model, loaders["val"], device)
        vm = kd.compute_metrics(val_logits, val_labels)
        is_best = early.step(vm["macro_f1"])
        history.append({"stage": stage, "epoch": epoch + 1, "train_loss": tr["loss"], "train_acc": tr["acc"],
                        "val_loss": vm["loss"], "val_top1": vm["top1"], "val_top3": vm["top3"],
                        "val_macro_f1": vm["macro_f1"], "lr": opt.param_groups[-1]["lr"],
                        "minutes": (time.time() - t0) / 60})
        h = history[-1]
        print(f"[{stage} {epoch + 1:>2}/{epochs}] train loss {h['train_loss']:.3f} acc {h['train_acc']:.3f} | "
              f"val loss {h['val_loss']:.3f} top1 {h['val_top1']:.3f} top3 {h['val_top3']:.3f} "
              f"F1 {h['val_macro_f1']:.3f} | {h['minutes']:.1f} min {'*BEST*' if is_best else ''}")
        if is_best:
            kd.save_checkpoint(ckpt_best, model=model, stage=stage, epoch=epoch, val=vm)
        stop = stage == "B" and early.should_stop
        kd.save_checkpoint(ckpt_last, model=model, optimizer=opt, scheduler=sched, scaler=scaler,
                           stage=stage, epoch=epoch, history=history, early=asdict(early),
                           done=stop or (stage == "B" and epoch + 1 == epochs))
        if stop:
            print(f"Early stopping: no val macro-F1 improvement for {PATIENCE} epochs.")
            break

run_stage("A", EPOCHS_HEAD, LR_STAGE_A, 0.0)
run_stage("B", EPOCHS_FT, LR_HEAD, LR_BACKBONE)

# Always continue with the BEST epoch's weights, not the last one
best = kd.load_checkpoint(ckpt_best, map_location=device, model=model)
print(f"\nLoaded best checkpoint: stage {best['stage']} epoch {best['epoch'] + 1}, val {best['val']}")

In [ ]:
hist = pd.DataFrame(history)
hist["global_epoch"] = range(1, len(hist) + 1)
hist.to_csv(OUT_DIR / "history.csv", index=False)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(hist.global_epoch, hist.train_loss, "o-", label="train loss (augmented)")
a1.plot(hist.global_epoch, hist.val_loss, "o-", label="val loss")
a2.plot(hist.global_epoch, hist.train_acc, "o-", label="train acc (augmented)")
a2.plot(hist.global_epoch, hist.val_top1, "o-", label="val top-1")
a2.plot(hist.global_epoch, hist.val_macro_f1, "o-", label="val macro-F1")
n_a = (hist.stage == "A").sum()
for a in (a1, a2):
    if 0 < n_a < len(hist):
        a.axvline(n_a + 0.5, color="grey", ls="--", lw=1)
    a.set_xlabel("epoch (dashed line: stage A | stage B)"); a.legend(); a.grid(alpha=0.3)
a1.set_title("Loss"); a2.set_title("Accuracy / F1")
plt.tight_layout(); plt.savefig(EXPORT_DIR / "training_curves.png", dpi=110); plt.show()
print(hist.round(4).to_string(index=False))

## 6. Calibration (temperature scaling, fitted on **val**)
The Bayesian Q&A engine uses these probabilities as its prior, so "90% sure" must really mean right about 90% of the time.

Both val and test use the same flip-TTA as the exported model.

In [ ]:
val_logits, val_labels = kd.predict(model, loaders["val"], device, tta=True)
T = kd.fit_temperature(val_logits, val_labels)
p_before, p_after = val_logits.softmax(1), (val_logits / T).softmax(1)
ece_before = kd.expected_calibration_error(p_before, val_labels)
ece_after = kd.expected_calibration_error(p_after, val_labels)
print(f"Temperature T = {T:.3f}   ({'model was over-confident' if T > 1 else 'model was under-confident'})")
print(f"val ECE: before {ece_before:.4f}  ->  after {ece_after:.4f}")
if not 0.5 <= T <= 5:
    print(f"WARNING: T={T:.2f} is unusual (typical 0.8-3). Expected in SMOKE_TEST; on a real run check the val set / training curves.")
val_metrics = kd.compute_metrics(val_logits, val_labels, temperature=T)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, probs, title in ((axes[0], p_before, f"Before (ECE {ece_before:.3f})"),
                         (axes[1], p_after, f"After T={T:.2f} (ECE {ece_after:.3f})")):
    bins = kd.reliability_bins(probs, val_labels)
    conf = [b[0] for b in bins if b[2]]; acc = [b[1] for b in bins if b[2]]; n = [b[2] for b in bins if b[2]]
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.bar(conf, acc, width=1 / len(bins), alpha=0.6, edgecolor="k", label="accuracy in bin")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title(title); ax.legend(loc="upper left")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.suptitle("Reliability diagram (validation set)")
plt.tight_layout(); plt.savefig(EXPORT_DIR / "reliability_diagram.png", dpi=110); plt.show()

## 7. Test evaluation: **run once**
This is the honest final score on images that were never used for any decision. **Don't tune anything after looking at it.**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_df = splits[splits.split == "test"].reset_index(drop=True)      # same order as loaders["test"]
test_logits, test_labels = kd.predict(model, loaders["test"], device, tta=True)
test_probs = (test_logits / T).softmax(1)
test_metrics = kd.compute_metrics(test_logits, test_labels, temperature=T)
test_metrics["ece"] = kd.expected_calibration_error(test_probs, test_labels)
test_pred = test_probs.argmax(1)

labels_idx = list(range(len(kd.CLASS_NAMES)))
report = classification_report(test_labels, test_pred, labels=labels_idx, target_names=kd.CLASS_NAMES,
                               digits=4, zero_division=0)
(EXPORT_DIR / "classification_report.txt").write_text(
    report + f"\ntop1={test_metrics['top1']:.4f} top3={test_metrics['top3']:.4f} "
             f"macro_f1={test_metrics['macro_f1']:.4f} ece={test_metrics['ece']:.4f} (test set, TTA, T={T:.3f})\n")
print(report)

recall = confusion_matrix(test_labels, test_pred, labels=labels_idx, normalize="true").diagonal()
old = json.loads(Path("../paddy_model/metrics.json").read_text()) if Path("../paddy_model/metrics.json").exists() else {}
print(f"{'':22}{'new (test)':>12}{'old (val*)':>12}{'target':>10}")
print(f"{'top-1':22}{test_metrics['top1']:>12.4f}{old.get('val_top1', float('nan')):>12.4f}{'>= 0.93':>10}")
print(f"{'top-3':22}{test_metrics['top3']:>12.4f}{old.get('val_top3', float('nan')):>12.4f}{'>= 0.98':>10}")
print(f"{'macro-F1':22}{test_metrics['macro_f1']:>12.4f}{0.7824:>12.4f}")
print(f"{'ECE (calibrated)':22}{test_metrics['ece']:>12.4f}{'n/a':>12}{'< 0.03':>10}")
print(f"{'min class recall':22}{recall.min():>12.4f}{'':>12}{'>= 0.85':>10}  ({kd.CLASS_NAMES[recall.argmin()]})")
print("* old model was scored on its own val set (also used for early stopping) -> optimistic.")
weak = [f"{c} ({r:.2f})" for c, r in zip(kd.CLASS_NAMES, recall) if r < 0.85]
print("Classes below 85% recall:", ", ".join(weak) if weak else "none")

In [ ]:
cm = confusion_matrix(test_labels, test_pred, labels=labels_idx)
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(labels_idx, kd.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticks(labels_idx, kd.CLASS_NAMES)
for i in labels_idx:
    for j in labels_idx:
        if cm[i, j]:
            ax.text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]:.0%}", ha="center", va="center", fontsize=7,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix (test, counts and row %)")
plt.colorbar(im, fraction=0.046); plt.tight_layout()
plt.savefig(EXPORT_DIR / "confusion_matrix.png", dpi=110); plt.show()

## 8. Error analysis
These are the most confident mistakes. Look at them:
- Some are **label errors** in the dataset.
- Some are genuinely **look-alike diseases**. Those are the pairs the follow-up Q&A engine has to tell apart by asking the farmer questions.

In [ ]:
pairs = [(cm[i, j], kd.CLASS_NAMES[i], kd.CLASS_NAMES[j]) for i in labels_idx for j in labels_idx if i != j and cm[i, j]]
print("Most confused pairs (true -> predicted):")
for n, t, p in sorted(pairs, reverse=True)[:8]:
    print(f"  {n:>3} x  {t:>26} -> {p}")

conf = test_probs.max(1).values
wrong = torch.nonzero(test_pred != test_labels).flatten()
wrong = wrong[conf[wrong].argsort(descending=True)][:16]
if len(wrong):
    cols = 4; rows_n = math.ceil(len(wrong) / cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.2, rows_n * 3.4), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, i in zip(axes.flat, wrong.tolist()):
        ax.imshow(Image.open(DATA_DIR / test_df.loc[i, "path"]).convert("RGB"))
        ax.set_title(f"true: {kd.CLASS_NAMES[test_labels[i]]}\npred: {kd.CLASS_NAMES[test_pred[i]]} ({conf[i]:.0%})", fontsize=8)
    plt.suptitle("Most confident mistakes (test)"); plt.tight_layout()
    plt.savefig(OUT_DIR / "worst_errors.png", dpi=90); plt.show()
else:
    print("No mistakes on the test set!")

## 9. Export to ONNX plus sidecar files
Flip-TTA is baked into `model.onnx`, so one call returns the averaged logits. The server then applies `softmax(logits / T)` using `temperature.json`.

In [ ]:
import onnx, onnxruntime as ort

export_model = kd.TTAWrapper(model).cpu().eval().float()
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
onnx_path = EXPORT_DIR / "model.onnx"
try:      # classic exporter: stable, opset 17
    torch.onnx.export(export_model, (dummy,), str(onnx_path), input_names=["image"], output_names=["logits"],
                      opset_version=17, dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, dynamo=False)
except Exception as e:   # newer torch without the classic exporter
    print("classic exporter failed:", e, "-> trying dynamo exporter")
    torch.onnx.export(export_model, (dummy,), str(onnx_path), input_names=["image"], output_names=["logits"],
                      dynamic_shapes=({0: torch.export.Dim("batch", min=1, max=64)},), dynamo=True)
onnx.checker.check_model(str(onnx_path))

# Check: onnxruntime must give the same numbers as PyTorch (on real test images)
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
x, _ = next(iter(loaders["test"]))
x = x[:8].float()
with torch.no_grad():
    ref = export_model(x).numpy()
got = sess.run(None, {"image": x.numpy()})[0]
max_diff = float(np.abs(ref - got).max())
print(f"onnx: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB) | max |torch - onnxruntime| = {max_diff:.2e}")
assert max_diff < 1e-4, "ONNX output differs from PyTorch!"
assert (ref.argmax(1) == got.argmax(1)).all()
model.to(device)

In [ ]:
eval_resize = IMG_SIZE
(EXPORT_DIR / "class_names.json").write_text(json.dumps(kd.CLASS_NAMES, indent=2))
(EXPORT_DIR / "preprocess.json").write_text(json.dumps({
    "input_name": "image", "output_name": "logits", "layout": "NCHW", "channel_order": "RGB", "dtype": "float32",
    "resize_shorter_side_to": eval_resize, "center_crop": IMG_SIZE, "interpolation": "bilinear (antialias)",
    "scale": "divide uint8 pixels by 255", "mean": list(MEAN), "std": list(STD),
    "tta_baked_in": "horizontal flip average", "postprocess": "softmax(logits / temperature)",
}, indent=2))
(EXPORT_DIR / "temperature.json").write_text(json.dumps({"temperature": T, "fitted_on": "val",
    "val_ece_before": ece_before, "val_ece_after": ece_after}, indent=2))
metrics_out = {
    "model_name": MODEL_NAME, "img_size": IMG_SIZE, "smoke_test": SMOKE_TEST,
    "best_epoch": {"stage": best["stage"], "epoch": best["epoch"] + 1},
    "test": test_metrics, "val_calibrated": val_metrics,
    "test_per_class_recall": dict(zip(kd.CLASS_NAMES, map(float, recall))),
    "n_images": counts[["train", "val", "test"]].sum().to_dict(),
    "onnx_max_abs_diff": max_diff, "temperature": T,
    "torch": torch.__version__, "timm": timm.__version__,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "baseline_old_model": {"val_top1": 0.7953, "val_top3": 0.9519, "val_macro_f1": 0.7824},
}
(EXPORT_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2, default=float))

zip_path = shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=EXPORT_DIR.parent, base_dir=EXPORT_DIR.name)
print("Export folder:", EXPORT_DIR.resolve())
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"  {f.name:28} {f.stat().st_size / 1e6:8.2f} MB")
print("\nSEND BACK:", Path(zip_path).resolve(), "+ this executed notebook")